In [ ]:
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

parquet_path = "/kaggle/input/datasets/tjaycuz/us-flight-data/US_flight_data.parquet"

pf = pq.ParquetFile(parquet_path)
print(f"Rows: {pf.metadata.num_rows:,} | Columns: {pf.metadata.num_columns}")

In [ ]:
md = pf.metadata
names = pf.schema_arrow.names
nulls = dict.fromkeys(names, 0)

for rg in range(md.num_row_groups):
    row_group = md.row_group(rg)
    for i, name in enumerate(names):
        stats = row_group.column(i).statistics
        if stats is not None and stats.has_null_count:
            nulls[name] += stats.null_count

null_df = (pd.DataFrame({"column": list(nulls), "nulls": list(nulls.values())})
             .assign(null_pct=lambda d: (d["nulls"] / md.num_rows * 100).round(2))
             .sort_values("null_pct", ascending=False)
             .reset_index(drop=True))
null_df.head(40)

In [ ]:
names = pf.schema_arrow.names
md = pf.metadata
nulls = dict.fromkeys(names, 0)
for rg in range(md.num_row_groups):
    r = md.row_group(rg)
    for i, n in enumerate(names):
        nulls[n] += r.column(i).statistics.null_count

total = md.num_rows
for i, n in enumerate(names):
    print(f"{i:3d}  {n:<28} {nulls[n]/total*100:6.2f}% null")

In [ ]:
drop_cols = [n for n in names if nulls[n] / total >= 0.99]
keep_cols = [n for n in names if n not in drop_cols]
print(f"Drop {len(drop_cols)} | Keep {len(keep_cols)}")
print(drop_cols)

In [ ]:
import time
text_cols = ["MKT_UNIQUE_CARRIER","BRANDED_CODE_SHARE","MKT_CARRIER","OP_UNIQUE_CARRIER","OP_CARRIER",
             "TAIL_NUM","ORIGIN","ORIGIN_CITY_NAME","ORIGIN_STATE_ABR","ORIGIN_STATE_NM",
             "DEST","DEST_CITY_NAME","DEST_STATE_ABR","DEST_STATE_NM","FL_DATE",
             "DEP_TIME_BLK","ARR_TIME_BLK","CANCELLATION_CODE","DUP"]
text_cols = [c for c in text_cols if c in keep_cols]
out_path = "/kaggle/working/flights_typed.parquet"

writer, t0, n = None, time.time(), 0
for batch in pf.iter_batches(batch_size=1_000_000, columns=keep_cols):
    arrays = [batch.column(c) if c in text_cols else pc.cast(batch.column(c), pa.float32()) for c in keep_cols]
    tbl = pa.Table.from_arrays(arrays, names=keep_cols)
    if writer is None:
        writer = pq.ParquetWriter(out_path, tbl.schema, compression="snappy")
    writer.write_table(tbl)
    n += len(tbl); print(f"{n:,} rows | {time.time()-t0:.0f}s")
writer.close()

In [ ]:
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

In [ ]:
df = pq.read_table(out_path, read_dictionary=text_cols).to_pandas()

In [ ]:
df.head()

In [ ]:
info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_%": (df.isna().mean() * 100).round(2),
    "nunique": df.nunique(),
    "MB": (df.memory_usage(deep=True) / 1e6).round(0),
})
print(f"Total: {info['MB'].sum()/1e3:.1f} GB")
info

In [ ]:
num_cols = df.select_dtypes("number").columns
info["min"] = df[num_cols].min()
info["max"] = df[num_cols].max()
info["range"] = info["min"].astype(str) + "  to  " + info["max"].astype(str)
info.loc[info["min"].isna(), "range"] = ""            # text/category columns
info["whole_nums"] = pd.Series({c: bool((df[c].dropna() % 1 == 0).all()) for c in num_cols})

pd.set_option("display.max_rows", None, "display.width", 200)
print(info[["dtype", "null_%", "nunique", "range", "whole_nums", "MB"]].to_string())

In [ ]:
i8  = ["QUARTER","MONTH","DAY_OF_MONTH","DAY_OF_WEEK","CANCELLED","DIVERTED","DISTANCE_GROUP",
       "DIV_AIRPORT_LANDINGS","FLIGHTS","ORIGIN_STATE_FIPS","DEST_STATE_FIPS","ORIGIN_WAC","DEST_WAC"]
i16 = ["YEAR","CRS_DEP_TIME","CRS_ARR_TIME","CRS_ELAPSED_TIME","DISTANCE","OP_CARRIER_FL_NUM",
       "OP_CARRIER_AIRLINE_ID","ORIGIN_AIRPORT_ID","DEST_AIRPORT_ID"]
u16 = ["ORIGIN_CITY_MARKET_ID","DEST_CITY_MARKET_ID"]              # max 36,422 > int16 limit
i32 = ["ORIGIN_AIRPORT_SEQ_ID","DEST_AIRPORT_SEQ_ID"]              # ~1.6M
# columns with nulls -> pandas nullable ints
n8  = ["DEP_DEL15","DEP_DELAY_GROUP","ARR_DEL15","ARR_DELAY_GROUP"]
n16 = ["DEP_TIME","ARR_TIME","WHEELS_OFF","WHEELS_ON","TAXI_OUT","TAXI_IN","AIR_TIME","ACTUAL_ELAPSED_TIME",
       "DEP_DELAY_NEW","ARR_DELAY","ARR_DELAY_NEW","CARRIER_DELAY","WEATHER_DELAY","NAS_DELAY",
       "SECURITY_DELAY","LATE_AIRCRAFT_DELAY","MKT_CARRIER_FL_NUM"]
nu16 = ["MKT_CARRIER_AIRLINE_ID"]

plan = {**dict.fromkeys(i8,"int8"), **dict.fromkeys(i16,"int16"), **dict.fromkeys(u16,"uint16"),
        **dict.fromkeys(i32,"int32"), **dict.fromkeys(n8,"Int8"), **dict.fromkeys(n16,"Int16"),
        **dict.fromkeys(nu16,"UInt16")}

before = df.memory_usage(deep=True).sum()/1e9
for c, t in plan.items():
    df[c] = df[c].astype(t)
gc.collect()
print(f"{before:.1f} GB -> {df.memory_usage(deep=True).sum()/1e9:.1f} GB")
print(df.dtypes.value_counts())

In [ ]:
bad = [c for c, t in plan.items() if t[0].islower() and df[c].isna().any()]
print("Columns with hidden nulls:", bad)
for c in bad:
    plan[c] = plan[c].replace("uint", "UInt").replace("int", "Int")   # int16 -> Int16, uint16 -> UInt16

before = df.memory_usage(deep=True).sum()/1e9
for c, t in plan.items():
    if str(df[c].dtype) != t:
        df[c] = df[c].astype(t)
gc.collect()
print(f"{before:.1f} GB -> {df.memory_usage(deep=True).sum()/1e9:.1f} GB")
print(df.dtypes.value_counts())

In [ ]:
print(pd.crosstab(df["DEP_DELAY"].isna(), [df["CANCELLED"], df["DIVERTED"]], margins=True))

In [ ]:

df.dtypes

In [ ]:
df.to_parquet("/kaggle/working/flights_df_68cols.parquet", compression="snappy", index=False)
import os; print(f"{os.path.getsize('/kaggle/working/flights_df_68cols.parquet')/1e9:.2f} GB saved")

In [ ]:
from IPython.display import FileLink
FileLink("flights_df_68cols.parquet")

----------------------------------------------------------------------------------------------------

In [ ]:
import pandas as pd

path = "/kaggle/input/datasets/tjaycuz/68-col-v2/flights_df_68cols (1).parquet"
df = pd.read_parquet(path)
print(df.shape)

num = df.select_dtypes("number")
summary = pd.DataFrame({"dtype": df.dtypes.astype(str)})
summary["min"] = num.min()
summary["max"] = num.max()
summary["nulls"] = df.isna().sum()
pd.set_option("display.max_rows", None)
summary

In [ ]:
pd.set_option("display.max_rows", None)

summary

In [ ]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"].astype(str))
print(df["FL_DATE"].min(), df["FL_DATE"].max())

bad = df[df["CRS_ELAPSED_TIME"] <= 0]
print(len(bad))
bad[["FL_DATE","OP_UNIQUE_CARRIER","ORIGIN","DEST","CRS_DEP_TIME","CRS_ARR_TIME","CRS_ELAPSED_TIME","DISTANCE"]].head(20)

----------------------------------------------------------------------------------------------------

### Project begins

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [8]:
path = "/kaggle/input/datasets/tjaycuz/68-col-v2/flights_df_68cols (1).parquet"
df = pd.read_parquet(path)




In [9]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 250)
pd.options.display.float_format = '{:,.2f}'.format


df.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,MKT_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_FIPS,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,DUP,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
0,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1,AA,19805,AA,N106NN,1,12478,1247805,31703,JFK,"New York, NY",NY,36,New York,22,12892,1289208,32575,LAX,"Los Angeles, CA",CA,6,California,91,700,657,-3.00,0,0,-1,0700-0759,19,716,1003,9,1029,1012,-17,0,0,-2,1000-1059,0,NaN,0,N,389,375,347,1,2475,10,<NA>,<NA>,<NA>,<NA>,<NA>,0
1,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,10,AA,19805,AA,N101NN,10,12892,1289208,32575,LAX,"Los Angeles, CA",CA,6,California,91,12478,1247805,31703,JFK,"New York, NY",NY,36,New York,22,900,857,-3.00,0,0,-1,0900-0959,20,917,1647,16,1724,1703,-21,0,0,-2,1700-1759,0,NaN,0,N,324,306,270,1,2475,10,<NA>,<NA>,<NA>,<NA>,<NA>,0
2,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1003,AA,19805,AA,N830NN,1003,15016,1501606,31123,STL,"St. Louis, MO",MO,29,Missouri,64,13930,1393007,30977,ORD,"Chicago, IL",IL,17,Illinois,41,705,<NA>,NaN,<NA>,<NA>,<NA>,0700-0759,<NA>,<NA>,<NA>,<NA>,831,<NA>,<NA>,<NA>,<NA>,<NA>,0800-0859,1,B,0,N,86,<NA>,<NA>,1,258,2,<NA>,<NA>,<NA>,<NA>,<NA>,0
3,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1004,AA,19805,AA,N805NN,1004,13930,1393007,30977,ORD,"Chicago, IL",IL,17,Illinois,41,11697,1169706,32467,FLL,"Fort Lauderdale, FL",FL,12,Florida,33,1830,937,907.00,907,1,12,1800-1859,23,1000,1346,3,2239,1349,910,910,1,12,2200-2259,0,NaN,0,N,189,192,166,1,1182,5,907,0,3,0,0,0
4,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1007,AA,19805,AA,N870NN,1007,11057,1105703,31057,CLT,"Charlotte, NC",NC,37,North Carolina,36,15016,1501606,31123,STL,"St. Louis, MO",MO,29,Missouri,64,2240,2300,20.00,20,1,1,2200-2259,16,2316,2400,4,2345,4,19,19,1,1,2300-2359,0,NaN,0,N,125,124,104,1,575,3,16,0,0,0,3,0


In [10]:
df.tail()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,MKT_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_FIPS,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,DUP,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
32957866,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4351,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,1751,1748,-3.00,0,0,-1,1700-1759,15,1803,1932,6,2000,1938,-22,0,0,-2,2000-2059,0,NaN,0,NaN,129,110,89,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957867,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4567,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,1459,1450,-9.00,0,0,-1,1400-1459,38,1528,1710,9,1719,1719,0,0,0,0,1700-1759,0,NaN,0,NaN,140,149,102,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957868,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4602,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,905,910,5.00,5,0,0,0900-0959,29,939,1110,4,1126,1114,-12,0,0,-1,1100-1159,0,NaN,0,NaN,141,124,91,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957869,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4664,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,1200,1158,-2.00,0,0,-1,1200-1259,13,1211,1349,6,1408,1355,-13,0,0,-1,1400-1459,0,NaN,0,NaN,128,117,98,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957870,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4809,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,609,603,-6.00,0,0,-1,0600-0659,16,619,804,8,825,812,-13,0,0,-1,0800-0859,0,NaN,0,NaN,136,129,105,1,762,4,<NA>,<NA>,<NA>,<NA>,<NA>,0


In [11]:
df.shape

(32957871, 68)

In [12]:
df.describe().style.set_sticky(axis="index")

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER_FL_NUM,OP_CARRIER_AIRLINE_ID,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN_STATE_FIPS,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST_STATE_FIPS,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
count,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,11845962.000000,11845962.000000,32957871.000000,32957870.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32392215.000000,32390910.000000,32390910.000000,32390910.000000,32390910.000000,32374792.000000,32374792.000000,32362463.000000,32362463.000000,32957871.000000,32362469.000000,32284610.000000,32284610.000000,32284610.000000,32284610.000000,32957871.000000,32957871.000000,32957861.000000,32284610.000000,32284610.000000,32957871.000000,32957871.000000,32957871.000000,6941163.000000,6941163.000000,6941163.000000,6941163.000000,6941163.000000,32957863.000000
mean,2023.857307,2.430481,6.273016,15.754513,3.990561,19828.266721,2474.739511,19956.779346,2492.955654,12662.609757,1266265.041996,31739.837376,27.087638,54.758899,12662.579057,1266261.971984,31739.834060,27.087552,54.758564,1327.753833,1331.506906,13.142225,16.389741,0.214261,0.204403,17.933889,1354.340872,1456.646098,8.340598,1490.572334,1459.792367,7.609666,16.295290,0.214999,-0.072283,0.017793,0.002635,145.998520,140.599632,114.340576,1.000000,826.295954,3.777731,24.945892,4.218339,13.680416,0.123574,28.112539,0.003607
std,1.354137,1.085856,3.330158,8.781800,2.007302,281.673957,1699.935652,371.705335,1656.588743,1530.925846,153092.393942,1322.726720,16.755187,26.628200,1530.926429,153092.452216,1322.711541,16.755083,26.628289,492.570057,509.388840,54.072216,55.402941,0.410309,2.395796,9.981859,511.886269,541.397807,7.003157,519.641475,546.925510,58.423693,55.085893,0.410822,2.549529,0.132197,0.051266,72.575472,72.677721,70.652747,0.000000,597.943819,2.345264,76.131305,33.745399,33.268729,3.448187,61.600948,0.106017
min,2022.000000,1.000000,1.000000,1.000000,1.000000,19393.000000,1.000000,19393.000000,1.000000,10135.000000,1013506.000000,30070.000000,1.000000,1.000000,10135.000000,1013506.000000,30070.000000,1.000000,1.000000,1.000000,1.000000,-115.000000,0.000000,0.000000,-2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-128.000000,0.000000,0.000000,-2.000000,0.000000,0.000000,-272.000000,14.000000,5.000000,1.000000,11.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2023.000000,1.000000,3.000000,8.000000,2.000000,19790.000000,1083.000000,19790.000000,1146.000000,11292.000000,1129202.000000,30647.000000,12.000000,34.000000,11292.000000,1129202.000000,30647.000000,12.000000,34.000000,907.000000,909.000000,-6.000000,0.000000,0.000000,-1.000000,12.000000,926.000000,1040.000000,4.000000,1102.000000,1043.000000,-15.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,92.000000,87.000000,63.000000,1.000000,391.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2024.000000,2.000000,6.000000,16.000000,4.000000,19805.000000,2158.000000,19930.000000,2213.000000,12889.000000,1288904.000000,31453.000000,26.000000,45.000000,12889.000000,1288904.000000,31453.000000,26.000000,45.000000,1320.000000,1324.000000,-2.000000,0.000000,0.000000,-1.000000,15.000000,1338.000000,1458.000000,6.000000,1516.000000,1502.000000,-6.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,129.000000,124.000000,97.000000,1.000000,674.000000,3.000000,4.000000,0.000000,0

## Functions

In [13]:
"""
basically print the average of all the flights form the year 2022 till 2026 
single average is the output for all the flights, (basically 32957871 / 12)

"""



def calculate_avg_flights_per_month(df):
    count = len(df["MONTH"])
    result = count/12
    return result


avg = calculate_avg_flights_per_month(df)
print(avg)

2746489.25


In [14]:
count = len(df["MONTH"])
print(count)

32957871


In [15]:
df["MONTH"].value_counts().sort_index()

MONTH
1     2768635
2     2595323
3     3011517
4     2941040
5     3046706
6     3045918
7     3155796
8     3083677
9     2294790
10    2415631
11    2277238
12    2321600
Name: count, dtype: int64

## Functions using Groupby

In [16]:

"""
Q . calculates the average number of flight records per year

Hint for groupby syntax is "df.groupby("Categorical_Column")["Numerical_Column"].operation()"
CATEGORICAL col FIRST and then the NUMERICAL cols comes.


Local and Global varibale. Local cannot be accessed outside our function. Global variable can be.
"""



def avg_flights_per_year(df):
    yearly_flights = df.groupby("YEAR").size()
    avg = yearly_flights.mean()
    return avg


avg = avg_flights_per_year(df)
print(avg)
                             
    

print(f" We have {avg} flights per year")
    

6591574.2
 We have 6591574.2 flights per year


In [17]:
yearly_flights = df.groupby("YEAR").size()
print(yearly_flights)
print(type(yearly_flights))   

YEAR
2022    7013508
2023    7134931
2024    7079081
2025    7001619
2026    4728732
dtype: int64
<class 'pandas.core.series.Series'>


In [18]:
"""
Q. average number of flight records per month across all years

"""


def avg_flights_per_month_across_all_years(df):
    month = df.groupby("MONTH").size()
    avg = month.mean()
    return avg

avg = avg_flights_per_month_across_all_years(df)
print(avg)

2746489.25


In [19]:
a=df.groupby("MONTH").size()
print(a)

MONTH
1     2768635
2     2595323
3     3011517
4     2941040
5     3046706
6     3045918
7     3155796
8     3083677
9     2294790
10    2415631
11    2277238
12    2321600
dtype: int64


In [20]:
df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'MKT_UNIQUE_CARRIER', 'BRANDED_CODE_SHARE', 'MKT_CARRIER_AIRLINE_ID', 'MKT_CARRIER', 'MKT_CARRIER_FL_NUM', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER',
       'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_FIPS', 'ORIGIN_STATE_NM', 'ORIGIN_WAC', 'DEST_AIRPORT_ID',
       'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_FIPS', 'DEST_STATE_NM', 'DEST_WAC', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'DEP_DELAY_GROUP',
       'DEP_TIME_BLK', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'ARR_DELAY_GROUP', 'ARR_TIME_BLK', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'DUP',
       'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32957871 entries, 0 to 32957870
Data columns (total 68 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   YEAR                    int16   
 1   QUARTER                 int8    
 2   MONTH                   int8    
 3   DAY_OF_MONTH            int8    
 4   DAY_OF_WEEK             int8    
 5   FL_DATE                 category
 6   MKT_UNIQUE_CARRIER      category
 7   BRANDED_CODE_SHARE      category
 8   MKT_CARRIER_AIRLINE_ID  UInt16  
 9   MKT_CARRIER             category
 10  MKT_CARRIER_FL_NUM      Int16   
 11  OP_UNIQUE_CARRIER       category
 12  OP_CARRIER_AIRLINE_ID   int16   
 13  OP_CARRIER              category
 14  TAIL_NUM                category
 15  OP_CARRIER_FL_NUM       Int16   
 16  ORIGIN_AIRPORT_ID       int16   
 17  ORIGIN_AIRPORT_SEQ_ID   int32   
 18  ORIGIN_CITY_MARKET_ID   uint16  
 19  ORIGIN                  category
 20  ORIGIN_CITY_NAME        category
 21  ORIGIN

### Data conversion done for FL_DATE col from category to datetime

In [22]:
df["FL_DATE"] = pd.to_datetime(
    df["FL_DATE"].astype(str),
    format="%m/%d/%Y %I:%M:%S %p"
)

print(df["FL_DATE"].dtype)

datetime64[ns]


In [23]:
d = df["DEP_DELAY"]; ok = d.notna()
print((df.loc[ok, "DEP_DELAY_NEW"].astype("float32") == d[ok].clip(lower=0)).all())
print((df.loc[ok, "DEP_DEL15"].astype("float32") == (d[ok] >= 15).astype("float32")).all())

a = df["ARR_DELAY"].astype("float32"); ok2 = a.notna()
print((df.loc[ok2, "ARR_DELAY_NEW"].astype("float32") == a[ok2].clip(lower=0)).all())
print((df.loc[ok2, "ARR_DEL15"].astype("float32") == (a[ok2] >= 15).astype("float32")).all())

True
True
True
True


In [24]:
has = df["MKT_UNIQUE_CARRIER"].notna()
print(df.groupby("YEAR")["MKT_UNIQUE_CARRIER"].apply(lambda s: s.notna().mean()).round(3))

sub = df[has]
codeshare = sub["MKT_UNIQUE_CARRIER"].astype(str) != sub["OP_UNIQUE_CARRIER"].astype(str)
print(codeshare.mean())
print(sub.groupby(codeshare)["DEP_DELAY"].mean())

YEAR
2022   1.00
2023   0.68
2024   0.00
2025   0.00
2026   0.00
Name: MKT_UNIQUE_CARRIER, dtype: float64
0.3018698692432071
False   14.89
True     9.40
Name: DEP_DELAY, dtype: float32


## Dropped Columns (68 → 55)

13 columns dropped: all are constants, rebuildable
from kept columns, or missing for most of the timeline.

**Constants (2)**
- `FLIGHTS`, `DUP`: a single unique value in every
  row, so they carry no information.

**Derived from the delay columns (6)**
- `DEP_DELAY_NEW`, `DEP_DEL15`, `DEP_DELAY_GROUP`:
  computed from `DEP_DELAY` (the target).
- `ARR_DELAY_NEW`, `ARR_DEL15`, `ARR_DELAY_GROUP`:
  computed from `ARR_DELAY`.
- Verified that the `NEW` and `DEL15` versions match
  their formulas on every row. All can be rebuilt in
  one line if needed.
- The departure versions are also target leakage if
  left in the feature set.

**Marketing carrier columns (5)**
- `MKT_UNIQUE_CARRIER`, `BRANDED_CODE_SHARE`,
  `MKT_CARRIER_AIRLINE_ID`, `MKT_CARRIER`,
  `MKT_CARRIER_FL_NUM`
- Filled for 100% of rows in 2022, 68% in 2023, and
  0% from 2024 onward (schema change in the BTS data),
  so they can't be used with a time-based split.
- `OP_UNIQUE_CARRIER` (operating carrier) is complete
  for all years and is used instead.

In [25]:
rebuildable = ["FLIGHTS", "DUP",
               "DEP_DELAY_NEW", "DEP_DEL15", "DEP_DELAY_GROUP",
               "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP"]

mkt = ["MKT_UNIQUE_CARRIER", "BRANDED_CODE_SHARE", "MKT_CARRIER_AIRLINE_ID",
       "MKT_CARRIER", "MKT_CARRIER_FL_NUM"]

df = df.drop(columns=rebuildable + mkt)
print(df.shape)

(32957871, 55)


 ### duplicates

In [26]:
key = ["FL_DATE","OP_UNIQUE_CARRIER","OP_CARRIER_FL_NUM","ORIGIN","DEST","CRS_DEP_TIME"]
print("dups:", df.duplicated(subset=key).sum())

dups: 0


### Setting lower and upper bounds

In [27]:
print(df["TAXI_OUT"].describe())
print(df["TAXI_OUT"].quantile([0.5, 0.99, 0.999, 0.9999]))
print("> 180 min:", (df["TAXI_OUT"] > 180).sum())

count   32,374,792.00
mean            17.93
std              9.98
min              1.00
25%             12.00
50%             15.00
75%             21.00
max          1,274.00
Name: TAXI_OUT, dtype: Float64
0.50     15
0.99     57
1.00    104
1.00    147
Name: TAXI_OUT, dtype: Int16
> 180 min: 90


In [29]:
# =====================================================================
# CLIPPING (Decision 7, revised in Session 6): run right after load +
# FL_DATE fix, BEFORE any feature building.
# =====================================================================
# Problem: DEP_DELAY has extreme outliers (max 7,223 min, about 5 days). 
# A few freak flights create huge errors and pull the model away from normal patterns.
# DEP_DELAY / ARR_DELAY: cap at 24 h (1,440 min), no lower cap.
#   A delay longer than a day is a cancellation in disguise and
#   unpredictable. ~2,624 flights affected.
# TAXI_OUT: cap at 180 min. 99.99% of flights are under 147 min; only 90
#   flights exceed 180 (max 1,274 = data error or stuck at the gate).
#   TAXI_OUT feeds PREV_TAXI_OUT, so freak values would leak into features.
# Clipping IN PLACE means the target and every delay/taxi-based feature
# are built from clipped values.
# =====================================================================

UPPER_CLIP = 1440
TAXI_CLIP = 180

# Keep the original DEP_DELAY for honest raw metrics. The guard makes the
# cell safe to re-run (otherwise a 2nd run would copy already-clipped data).
if "DEP_DELAY_RAW" not in df.columns:
    df["DEP_DELAY_RAW"] = df["DEP_DELAY"].astype("float32")

# Clip in place. NaN (cancelled flights) stays NaN, so the TAIL_NUM chain
# for lag features is intact. float32 keeps NaN and saves RAM.
for col in ["DEP_DELAY", "ARR_DELAY"]:
    df[col] = df[col].astype("float32").clip(upper=UPPER_CLIP)
df["TAXI_OUT"] = df["TAXI_OUT"].astype("float32").clip(upper=TAXI_CLIP)

# Verify
m = df["CANCELLED"] == 0
print("DEP_DELAY rows changed:", (df.loc[m, "DEP_DELAY"] != df.loc[m, "DEP_DELAY_RAW"]).sum())  # ~2,624
print("DEP_DELAY max:", df["DEP_DELAY"].max())    # 1440.0
print("ARR_DELAY max:", df["ARR_DELAY"].max())    # <= 1440.0
print("TAXI_OUT max:", df["TAXI_OUT"].max())      # 180.0
print("NaN count unchanged:", df["DEP_DELAY"].isna().sum() == df["DEP_DELAY_RAW"].isna().sum())  # True

DEP_DELAY rows changed: 2624
DEP_DELAY max: 1440.0
ARR_DELAY max: 1440.0
TAXI_OUT max: 180.0
NaN count unchanged: True


In [1]:
import glob
import pyarrow.parquet as pq
import pandas as pd

# 1) Locate both datasets (old = merged US file from the earlier project, new = Jun-Aug 2026 fetch)
old_paths = glob.glob("/kaggle/input/**/*us_features_v1_with_weather*.parquet", recursive=True)
new_paths = glob.glob("/kaggle/input/**/weather_us_*.parquet", recursive=True)
print("old:", old_paths)
print("new:", new_paths)

# 2) OLD file: schema + row count only, so we learn the real column names without loading 30M rows
pf = pq.ParquetFile(old_paths[0])
print("\nold rows:", pf.metadata.num_rows)
print(pf.schema_arrow)

# 3) NEW file: small, so load fully and sanity-check it
new = pd.read_parquet(new_paths[0])
print("\nnew shape:", new.shape)
print(new.dtypes)
print("airports:", new["airport"].nunique(),
      "| range:", new["datetime"].min(), "->", new["datetime"].max())
# expect 2208 rows for every airport (92 days x 24 h)
print("rows per airport:", new.groupby("airport").size().value_counts().to_dict())
print("dup (airport, datetime):", new.duplicated(["airport", "datetime"]).sum())   # expect 0
print("missing share per column:")
print(new.drop(columns=["airport", "datetime", "weather_station_id"]).isna().mean().round(3))

# 4) Do we have weather for every origin airport in our flight data? (only if df is loaded)
if "df" in globals():
    missing = sorted(set(df["ORIGIN"].astype(str).unique()) - set(new["airport"]))
    print("\nORIGIN airports with no new weather:", len(missing), missing[:30])

old: ['/kaggle/input/datasets/tjaycuz/us-features-v1-with-weather-parquet/us_features_v1_with_weather.parquet']
new: []

old rows: 30538520
airline: string
flight_number: int64
origin_airport: string
destination_airport: string
scheduled_departure: timestamp[ns]
actual_departure: string
scheduled_arrival: string
actual_arrival: string
flight_status: string
cancellation_reason: string
flight_type: string
country: string
cancellation_reason_available: bool
departure_delay_minutes: double
has_departure_delay_target: bool
arrival_delay_minutes: double
scheduled_flight_duration_minutes: double
hour_of_day: int64
day_of_week: int64
month: int64
is_weekend: bool
is_redeye: bool
route: string
weather_hour: timestamp[ns]
temperature_2m_c: double
windspeed_10m_ms: double
windgusts_10m_ms: double
visibility_m: double
precipitation_mm: double
snowfall_cm: double
cloudcover_low_pct: double
weathercode: double
weather_station_id: string
is_substitute_station: int64
station_distance_km: double
-- sch

IndexError: list index out of range

In [25]:
# =====================================================================
# LAG FEATURES: what the SAME PLANE (TAIL_NUM) did on its previous flight
# =====================================================================
# Cancelled rows are NOT dropped here, so the chain per plane stays intact.
# Work on a small copy (only the columns needed), then attach results back to df.

lag = df[["TAIL_NUM", "SCHED_DEP_DT", "CRS_DEP_TIME", "CRS_ARR_TIME",
          "CRS_ELAPSED_TIME", "ARR_DELAY", "TAXI_OUT", "DEP_DELAY",
          "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID"]].copy()

# STEP 1: scheduled arrival as a local-time timestamp.
# Clock difference between arrival and departure minus the scheduled flight time
# gives the time zone shift (a multiple of 60 min, wrapped around midnight).
dep_clk = (lag["CRS_DEP_TIME"].astype("int32") // 100) * 60 + lag["CRS_DEP_TIME"].astype("int32") % 100
arr_clk = (lag["CRS_ARR_TIME"].astype("int32") // 100) * 60 + lag["CRS_ARR_TIME"].astype("int32") % 100
elapsed = lag["CRS_ELAPSED_TIME"].astype("float32")
tz_shift = (((arr_clk - dep_clk - elapsed) + 720) % 1440) - 720
sched_arr = lag["SCHED_DEP_DT"] + pd.to_timedelta(elapsed + tz_shift, unit="m")

# STEP 2: ACTUAL arrival timestamp = scheduled arrival + ARR_DELAY (NaT if cancelled/diverted)
lag["ACT_ARR_DT"] = sched_arr + pd.to_timedelta(lag["ARR_DELAY"].astype("float32"), unit="m")
del dep_clk, arr_clk, elapsed, tz_shift, sched_arr

# STEP 3: sort by plane, then by scheduled departure; take the previous row per plane
lag = lag.sort_values(["TAIL_NUM", "SCHED_DEP_DT"])
g = lag.groupby("TAIL_NUM", observed=True)
prev_arr_dt    = g["ACT_ARR_DT"].shift(1)
prev_arr_delay = g["ARR_DELAY"].shift(1)
prev_taxi_out  = g["TAXI_OUT"].shift(1)
prev_dep_delay = g["DEP_DELAY"].shift(1)
prev_dest      = g["DEST_AIRPORT_ID"].shift(1)

# STEP 4: validity. The previous flight must have landed BEFORE this scheduled
# departure, at THIS flight's origin, and not more than 24 h earlier (stale).
gap = (lag["SCHED_DEP_DT"] - prev_arr_dt).dt.total_seconds() / 60
valid = (gap >= 0) & (gap <= 1440) & (prev_dest == lag["ORIGIN_AIRPORT_ID"])

# STEP 5: attach back to df (same index), invalid rows become NaN
df["PREV_ARR_DELAY"]  = prev_arr_delay.where(valid).astype("float32")
df["PREV_TAXI_OUT"]   = prev_taxi_out.where(valid).astype("float32")
df["PREV_DEP_DELAY"]  = prev_dep_delay.where(valid).astype("float32")
df["TURNAROUND_MIN"]  = gap.where(valid).astype("float32")   # time on ground before this departure
del lag, g, prev_arr_dt, prev_arr_delay, prev_taxi_out, prev_dep_delay, prev_dest, gap, valid

# STEP 6: verify
print("share with a valid previous flight:", round(df["PREV_ARR_DELAY"].notna().mean() * 100, 2), "%")
print(df[["PREV_ARR_DELAY", "PREV_TAXI_OUT", "PREV_DEP_DELAY", "TURNAROUND_MIN"]].describe())
m = df["CANCELLED"] == 0
print("corr PREV_ARR_DELAY vs DEP_DELAY_CLIP:",
      round(df.loc[m, "PREV_ARR_DELAY"].corr(df.loc[m, "DEP_DELAY_CLIP"]), 3))

share with a valid previous flight: 89.57 %
       PREV_ARR_DELAY  PREV_TAXI_OUT  PREV_DEP_DELAY  TURNAROUND_MIN
count   29,520,551.00  29,520,551.00   29,520,551.00   29,520,551.00
mean             1.31          17.57            7.34          191.90
std             39.60           8.72           37.96          233.18
min           -128.00           1.00          -99.00            0.00
25%            -16.00          12.00           -6.00           52.00
50%             -7.00          15.00           -2.00           74.00
75%              7.00          20.00            7.00          207.00
max          4,381.00         379.00        4,380.00        1,440.00
corr PREV_ARR_DELAY vs DEP_DELAY_CLIP: 0.109


In [26]:
m = (df["CANCELLED"] == 0) & df["PREV_ARR_DELAY"].notna()
d = df.loc[m, ["PREV_ARR_DELAY", "DEP_DELAY_CLIP", "TURNAROUND_MIN"]]
d["bucket"] = pd.cut(d["TURNAROUND_MIN"], [-1, 45, 60, 90, 120, 240, 1440])
print(d.groupby("bucket", observed=True).apply(
    lambda x: pd.Series({"rows": len(x), "corr": x["PREV_ARR_DELAY"].corr(x["DEP_DELAY_CLIP"])})))

                    rows  corr
bucket                        
(-1, 45]    5,158,353.00  0.18
(45, 60]    5,086,504.00  0.09
(60, 90]    8,189,912.00  0.05
(90, 120]   2,356,158.00  0.05
(120, 240]  1,446,621.00  0.05
(240, 1440] 7,177,008.00  0.05


/tmp/ipykernel_49/3542446300.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(d.groupby("bucket", observed=True).apply(


In [27]:
# CHECK: does a late previous arrival lead to a late departure? (turnaround <= 90 min)
m = (df["CANCELLED"] == 0) & df["PREV_ARR_DELAY"].notna() & (df["TURNAROUND_MIN"] <= 90)
d = df.loc[m, ["PREV_ARR_DELAY", "DEP_DELAY_CLIP"]]
d["prev_arr_bin"] = pd.cut(d["PREV_ARR_DELAY"], [-1000, -30, -15, 0, 15, 30, 60, 120, 5000])

out = d.groupby("prev_arr_bin", observed=True)["DEP_DELAY_CLIP"].agg(
    rows="size",
    mean_dep_delay="mean",
    median_dep_delay="median",
    share_over_15=lambda s: (s > 15).mean())
print(out.round(2))

# Rank correlation (ignores outliers), on a 2M sample for speed
s = d.sample(2_000_000, random_state=0)
print("Spearman:", round(s["PREV_ARR_DELAY"].corr(s["DEP_DELAY_CLIP"], method="spearman"), 3))

                 rows  mean_dep_delay  median_dep_delay  share_over_15
prev_arr_bin                                                          
(-1000, -30]   307511            0.60             -5.00           0.10
(-30, -15]    4177017           -0.05             -5.00           0.07
(-15, 0]      7728743            2.12             -3.00           0.09
(0, 15]       3522280            9.59              3.00           0.21
(15, 30]      1478348           22.25             18.00           0.56
(30, 60]       956792           36.31             32.00           0.80
(60, 120]      187574           36.91             31.00           0.69
(120, 5000]     76504           29.30             18.00           0.54
Spearman: 0.487


In [28]:
# =====================================================================
# MODEL MATRIX + TIME-BASED SPLIT
# =====================================================================
# Lag features are built, so cancelled rows can go now.
data = df[df["CANCELLED"] == 0].copy()
data = data[data["DEP_DELAY"].notna()]          # also covers the 19,447 cancelled-with-delay rows
print("rows after dropping cancelled:", len(data))

# Calendar features
data["DOW"]   = data["FL_DATE"].dt.dayofweek.astype("int8")
data["MONTH"] = data["FL_DATE"].dt.month.astype("int8")
data["DAY_OF_YEAR"] = data["FL_DATE"].dt.dayofyear.astype("int16")

# Features: only things known BEFORE departure
FEATURES = [
    # time
    "CRS_DEP_HOUR", "DOW", "MONTH", "DAY_OF_YEAR",
    # route / carrier (categorical)
    "OP_UNIQUE_CARRIER", "ORIGIN", "DEST",
    # schedule
    "CRS_ARR_TIME", "CRS_ELAPSED_TIME", "DISTANCE",
    # lag features from the same plane's previous flight
    "PREV_ARR_DELAY", "PREV_TAXI_OUT", "PREV_DEP_DELAY", "TURNAROUND_MIN",
]
TARGET = "DEP_DELAY_CLIP"

# Time-based split: train up to 2026-05-31, test on Jun to Aug 2026
cut = pd.Timestamp("2026-06-01")
train = data[data["FL_DATE"] < cut]
test  = data[data["FL_DATE"] >= cut]
print("train:", len(train), "| test:", len(test))
print("train dates:", train["FL_DATE"].min().date(), "to", train["FL_DATE"].max().date())
print("test dates: ", test["FL_DATE"].min().date(), "to", test["FL_DATE"].max().date())

# Cast

rows after dropping cancelled: 32371463
train: 30561389 | test: 1810074
train dates: 2022-01-01 to 2026-05-31
test dates:  2026-06-01 to 2026-08-31


In [29]:
import gc, lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# STEP 1: keep only what the model needs, free the big frames
keep = FEATURES + [TARGET, "DEP_DELAY", "FL_DATE"]
train = train[keep]; test = test[keep]
del df, data; gc.collect()

# STEP 2: validation = last month of train (for early stopping). Test stays untouched.
val_cut = pd.Timestamp("2026-05-01")
tr  = train[train["FL_DATE"] < val_cut]
val = train[train["FL_DATE"] >= val_cut]
print("fit:", len(tr), "| val:", len(val), "| test:", len(test))

# STEP 3: dumb baseline = predict the training median every time
base = tr[TARGET].median()
for name, y_raw, y_clip in [("test", test["DEP_DELAY"], test[TARGET])]:
    p = np.full(len(y_raw), base)
    print(f"BASELINE (always {base}): MAE raw {mean_absolute_error(y_raw, p):.2f} | "
          f"RMSE raw {mean_squared_error(y_raw, p)**0.5:.2f} | RMSE clip {mean_squared_error(y_clip, p)**0.5:.2f}")

# STEP 4: LightGBM
dtr  = lgb.Dataset(tr[FEATURES],  tr[TARGET],  categorical_feature=["OP_UNIQUE_CARRIER", "ORIGIN", "DEST"], free_raw_data=True)
dval = lgb.Dataset(val[FEATURES], val[TARGET], reference=dtr)

params = dict(objective="regression", metric="rmse", learning_rate=0.1,
              num_leaves=255, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.5, bagging_freq=1, cat_smooth=50,
              max_bin=255, num_threads=-1, verbose=-1)

model = lgb.train(params, dtr, num_boost_round=400, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(30), lgb.log_evaluation(25)])

# STEP 5: evaluate on the untouched test months
pred = model.predict(test[FEATURES], num_iteration=model.best_iteration)
for label, y in [("raw DEP_DELAY", test["DEP_DELAY"]), ("clipped target", test[TARGET])]:
    print(f"{label:15s} MAE {mean_absolute_error(y, pred):.2f} | "
          f"RMSE {mean_squared_error(y, pred)**0.5:.2f} | R2 {r2_score(y, pred):.3f}")

# STEP 6: what the model relies on
imp = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
print((imp / imp.sum() * 100).round(1))

fit: 29955309 | val: 606080 | test: 1810074
BASELINE (always -2.0): MAE raw 25.25 | RMSE raw 69.32 | RMSE clip 68.31
Training until validation scores don't improve for 30 rounds
[25]	valid_0's rmse: 46.7253
[50]	valid_0's rmse: 46.6107
[75]	valid_0's rmse: 46.5891
[100]	valid_0's rmse: 46.5706
[125]	valid_0's rmse: 46.5672
[150]	valid_0's rmse: 46.562
[175]	valid_0's rmse: 46.5572
[200]	valid_0's rmse: 46.5573
Early stopping, best iteration is:
[181]	valid_0's rmse: 46.556
raw DEP_DELAY   MAE 20.40 | RMSE 56.98 | R2 0.255
clipped target  MAE 20.36 | RMSE 55.77 | R2 0.263
TURNAROUND_MIN      52.60
PREV_TAXI_OUT       11.40
ORIGIN               9.20
DEST                 5.50
OP_UNIQUE_CARRIER    4.70
CRS_DEP_HOUR         3.40
PREV_ARR_DELAY       3.20
DAY_OF_YEAR          2.80
CRS_ARR_TIME         2.60
PREV_DEP_DELAY       1.70
DOW                  0.80
CRS_ELAPSED_TIME     0.80
DISTANCE             0.80
MONTH                0.30
dtype: float64


In [31]:
# =====================================================================
# ONE CELL: load -> clean -> leak-free features -> seasonal backtest -> R2
# Rule: every feature uses only info known at SCHEDULED departure time.
# =====================================================================
import gc, time, numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from pandas.tseries.holiday import USFederalHolidayCalendar

t0 = time.time()
def log(m): print(f"[{(time.time()-t0)/60:5.1f} min] {m}", flush=True)

# ---------------- CONFIG (change here) ----------------
PATH = "/kaggle/input/datasets/tjaycuz/68-col-v2/flights_df_68cols (1).parquet"
UPPER_CLIP = 1440            # target cap, 24 h (decision made earlier, no lower clip)
MAX_TRAIN_ROWS = 10_000_000  # most recent N rows per fold; None = use everything (slower)
FOLDS = [                    # expanding-window backtest: (name, test_start, test_end)
    ("fall_2025",   "2025-09-01", "2025-12-01"),
    ("winter_25_26","2025-12-01", "2026-03-01"),
    ("spring_2026", "2026-03-01", "2026-06-01"),
    ("summer_2026", "2026-06-01", "2026-09-01"),
]
# ------------------------------------------------------

# ---------- 1. LOAD only the columns we need ----------
COLS = ["FL_DATE", "TAIL_NUM", "OP_UNIQUE_CARRIER", "ORIGIN", "DEST",
        "ORIGIN_STATE_ABR", "DEST_STATE_ABR", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
        "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME", "DISTANCE",
        "DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "CANCELLED"]
df = pd.read_parquet(PATH, columns=COLS)
# FL_DATE labels look like "1/1/2022 12:00:00 AM" (month first). Parse only the unique
# labels (about 1,700), then map back to rows: fast and memory-light.
if str(df["FL_DATE"].dtype) == "category":
    labels = df["FL_DATE"].cat.categories.astype(str)
    try:
        parsed = pd.to_datetime(labels, format="%m/%d/%Y %I:%M:%S %p")
    except ValueError:
        parsed = pd.to_datetime(labels, format="mixed", dayfirst=False)
    df["FL_DATE"] = parsed.values[df["FL_DATE"].cat.codes.values]
else:
    df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

# Sanity check: must match the known range, otherwise day and month were swapped
print(df["FL_DATE"].min(), df["FL_DATE"].max())
assert df["FL_DATE"].min() == pd.Timestamp("2022-01-01") and df["FL_DATE"].max() == pd.Timestamp("2026-08-31"), \
    "FL_DATE range is wrong, check the date format"

n = len(df); log(f"loaded {n:,} rows")


# CRS_ELAPSED_TIME: <=0 -> NaN -> route median (same fix as Session 4)
el = df["CRS_ELAPSED_TIME"].astype("float32")
df["CRS_ELAPSED_TIME"] = el.where(el > 0)
med = df.groupby(["ORIGIN", "DEST"], observed=True)["CRS_ELAPSED_TIME"].transform("median")
df["CRS_ELAPSED_TIME"] = df["CRS_ELAPSED_TIME"].fillna(med)
del el, med

# ---------- 2. BASE ARRAYS (numpy = fast) ----------
dt = df["CRS_DEP_TIME"].astype("int32").to_numpy()
at = df["CRS_ARR_TIME"].astype("int32").to_numpy()
dep_clk = (dt // 100) * 60 + dt % 100                       # 2400 -> 1440 (next-day midnight)
arr_clk = (at // 100) * 60 + at % 100
day0    = df["FL_DATE"].to_numpy().astype("datetime64[m]").astype("int64")   # minutes since epoch
sdep    = day0 + dep_clk                                    # scheduled departure, origin-local minutes
elapsed = df["CRS_ELAPSED_TIME"].to_numpy()
tz      = (((arr_clk - dep_clk - elapsed) + 720) % 1440) - 720       # time zone shift
sarr    = sdep + elapsed + tz                               # scheduled arrival, DEST-local minutes
dd      = df["DEP_DELAY"].astype("float32").to_numpy()      # NaN = cancelled
ad      = df["ARR_DELAY"].astype("float32").to_numpy()
tx      = df["TAXI_OUT"].astype("float32").to_numpy()
canc    = df["CANCELLED"].to_numpy().astype("int8")
tail    = df["TAIL_NUM"].cat.codes.to_numpy().astype("int32")        # -1 = missing tail
oid     = df["ORIGIN_AIRPORT_ID"].to_numpy().astype("int32")
did     = df["DEST_AIRPORT_ID"].to_numpy().astype("int32")
oc      = df["ORIGIN"].cat.codes.to_numpy().astype("int64")
cc      = df["OP_UNIQUE_CARRIER"].cat.codes.to_numpy().astype("int64")
del dep_clk, arr_clk, tz, elapsed
feats = {}

# ---------- 3. PLANE CHAIN (what the same plane did before) ----------
idx = np.lexsort((sdep, tail))                # primary: plane, secondary: scheduled time
S = lambda a: a[idx]
tail_s, sdep_s, sarr_s = S(tail), S(sdep), S(sarr)
dd_s, ad_s, tx_s, canc_s = S(dd), S(ad), S(tx), S(canc)
oid_s, did_s, day_s = S(oid), S(did), S(day0)

def sh(a, k, fill):                           # value k rows earlier
    out = np.empty_like(a); out[:k] = fill; out[k:] = a[:-k]; return out

same1 = (tail_s == sh(tail_s, 1, -2)) & (tail_s >= 0)        # prev row is same plane
same2 = same1 & sh(same1, 1, False)                          # two rows back is same plane too
p_sdep, p_sarr = sh(sdep_s, 1, 0), sh(sarr_s, 1, np.nan)
p_dd, p_ad, p_tx = sh(dd_s, 1, np.nan), sh(ad_s, 1, np.nan), sh(tx_s, 1, np.nan)
p_canc, p_did = sh(canc_s, 1, 0), sh(did_s, 1, -1)

chain = same1 & (p_did == oid_s)                             # prev flight ended where this one starts
p_dep_act = p_sdep + p_dd                                    # when prev actually left
p_arr_act = p_sarr + p_ad                                    # when prev actually landed
dep_known = chain & (p_dep_act <= sdep_s)                    # prev had ALREADY left at our sched departure
tx_known  = chain & (p_dep_act + p_tx <= sdep_s)
arr_known = chain & (p_arr_act <= sdep_s)                    # prev had ALREADY landed

st = {}
st["SCHED_TURN"]     = np.where(chain, sdep_s - p_sarr, np.nan)                 # scheduled slack (always known)
st["INBOUND_SLACK"]  = np.where(dep_known, sdep_s - (p_sarr + p_dd), np.nan)    # expected slack given prev delay
st["PREV_DEP_DELAY"] = np.where(dep_known, p_dd, np.nan)
st["PREV_ARR_DELAY"] = np.where(arr_known, p_ad, np.nan)
st["PREV_TAXI_OUT"]  = np.where(tx_known, p_tx, np.nan)
st["TURN_ACT"]       = np.where(arr_known, sdep_s - p_arr_act, np.nan)
st["PREV_CANCELLED"] = np.where(chain, (p_canc == 1).astype("float32"), np.nan)
dep2_known = same2 & ((sh(sdep_s, 2, 0) + sh(dd_s, 2, np.nan)) <= sdep_s)
st["PREV2_DEP_DELAY"] = np.where(dep2_known, sh(dd_s, 2, np.nan), np.nan)
new_day = ~same1 | (day_s != sh(day_s, 1, -1))
start = np.maximum.accumulate(np.where(new_day, np.arange(n), 0))
st["LEG_OF_DAY"] = np.where(tail_s >= 0, np.arange(n) - start, np.nan)
status = np.where(~chain, 0, np.where(arr_known, 3, np.where(dep_known, 2, 1)))
# 0 no chain | 1 prev not departed/cancelled | 2 prev in the air | 3 prev landed
def back(a, dtype=np.float32):
    out = np.empty(n, dtype=dtype); out[idx] = a; return out
for k, v in st.items(): feats[k] = back(v)
feats["PREV_STATUS"] = back(status, np.float32)
del (S, tail_s, sdep_s, sarr_s, dd_s, ad_s, tx_s, canc_s, oid_s, did_s, day_s, same1, same2,
     p_sdep, p_sarr, p_dd, p_ad, p_tx, p_canc, p_did, chain, p_dep_act, p_arr_act,
     dep_known, tx_known, arr_known, dep2_known, new_day, start, status, st, idx)
gc.collect(); log("plane-chain features done")

# ---------- 4. AIRPORT / CARRIER CONGESTION (only flights that ALREADY departed) ----------
BIG = 200_000
sb   = sdep // 60                                   # scheduled hour bin, origin-local
hmin = sb.min() - 10
HB   = sb - hmin
ok   = ~np.isnan(dd)
ab   = np.floor((sdep[ok] + dd[ok]) / 60).astype("int64") - hmin   # ACTUAL departure hour bin

def lookup(uk, arr, q):
    pos = np.searchsorted(uk, q); pos[pos >= len(uk)] = len(uk) - 1
    return np.where(uk[pos] == q, arr[pos], np.nan).astype("float32")

# (a) origin airport: flights that actually departed in hour h-1, h-2, h-3
d = pd.DataFrame({"k": oc[ok] * BIG + ab, "dd": dd[ok], "late": (dd[ok] > 15).astype("float32"), "n": 1.0})
g = d.groupby("k").agg(dd=("dd", "mean"), late=("late", "mean"), n=("n", "sum"))
uk = g.index.values
for h in (1, 2, 3):
    feats[f"ORG_DELAY_H{h}"] = lookup(uk, g["dd"].values, oc * BIG + HB - h)
feats["ORG_LATE_H1"] = lookup(uk, g["late"].values, oc * BIG + HB - 1)
feats["ORG_N_H1"]    = lookup(uk, g["n"].values,    oc * BIG + HB - 1)

# (b) scheduled load and cancel rate (schedule is known in advance)
d = pd.DataFrame({"k": oc * BIG + HB, "c": canc.astype("float32"), "n": 1.0})
g = d.groupby("k").agg(c=("c", "mean"), n=("n", "sum")); uk = g.index.values
feats["SCHED_N_H0"]      = lookup(uk, g["n"].values, oc * BIG + HB)
feats["SCHED_N_H1"]      = lookup(uk, g["n"].values, oc * BIG + HB - 1)
feats["CANCEL_RATE_H1"]  = lookup(uk, g["c"].values, oc * BIG + HB - 1)
feats["ORG_LOAD_RATIO"]  = np.divide(feats["ORG_N_H1"], feats["SCHED_N_H1"],
                                     out=np.full(n, np.nan, "float32"), where=feats["SCHED_N_H1"] > 0)

# (c) same carrier at same origin, previous hour
d = pd.DataFrame({"k": (oc[ok] * 128 + cc[ok]) * BIG + ab, "dd": dd[ok], "n": 1.0})
g = d.groupby("k").agg(dd=("dd", "mean"), n=("n", "sum")); uk = g.index.values
q = (oc * 128 + cc) * BIG + HB - 1
feats["CAR_DELAY_H1"] = lookup(uk, g["dd"].values, q)
feats["CAR_N_H1"]     = lookup(uk, g["n"].values, q)
del d, g, uk, q, ab, sb; gc.collect(); log("congestion features done")

# ---------- 5. CALENDAR + SCHEDULE FEATURES ----------
fdate = df["FL_DATE"]
feats["DOW"]   = fdate.dt.dayofweek.to_numpy().astype("float32")
feats["MONTH"] = fdate.dt.month.to_numpy().astype("float32")
feats["DOY"]   = fdate.dt.dayofyear.to_numpy().astype("float32")
feats["DEP_HOUR"]    = ((dt // 100) % 24).astype("float32")
feats["DEP_MIN_DAY"] = (sdep - day0).astype("float32") % 1440
feats["ARR_HOUR"]    = ((at // 100) % 24).astype("float32")
feats["DISTANCE"]    = df["DISTANCE"].astype("float32").to_numpy()
feats["CRS_ELAPSED"] = df["CRS_ELAPSED_TIME"].to_numpy().astype("float32")
hol = USFederalHolidayCalendar().holidays("2021-12-01", "2026-12-31").values.astype("datetime64[D]").astype("int64")
days = day0 // 1440
pos = np.searchsorted(hol, days)
feats["DAYS_TO_HOLIDAY"] = np.minimum(np.abs(hol[np.clip(pos, 0, len(hol)-1)] - days),
                                      np.abs(days - hol[np.clip(pos-1, 0, len(hol)-1)])).astype("float32")
del dt, at, sdep, sarr, day0, days, pos, tail, oid, did, oc, cc, tx, ad; gc.collect()

# ---------- 6. FINAL MATRIX (only flown flights) ----------
keep = (canc == 0) & ok
CATS = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "ORIGIN_STATE_ABR", "DEST_STATE_ABR"]
data = {c: df[c].array[keep] for c in CATS}
for k in list(feats): data[k] = feats.pop(k)[keep]
X = pd.DataFrame(data); del data
y_raw  = dd[keep]
y      = np.minimum(y_raw, UPPER_CLIP)
fd     = df["FL_DATE"].to_numpy()[keep]
FEATURES = list(X.columns)
del df, feats; gc.collect()
log(f"matrix ready: {X.shape[0]:,} rows x {X.shape[1]} features")

# ---------- 7. SEASONAL BACKTEST (expanding window, never trains on the future) ----------
params = dict(objective="regression", metric="rmse", learning_rate=0.1, num_leaves=255,
              min_data_in_leaf=500, feature_fraction=0.7, bagging_fraction=0.6, bagging_freq=1,
              lambda_l2=10, max_bin=127, cat_smooth=50, cat_l2=10, num_threads=-1, verbose=-1)
results = []
for name, t_start, t_end in FOLDS:
    ts, te = np.datetime64(t_start), np.datetime64(t_end)
    vs = ts - np.timedelta64(28, "D")                       # last 28 days before test = early-stopping set
    tr_i = np.flatnonzero(fd < vs)
    va_i = np.flatnonzero((fd >= vs) & (fd < ts))
    te_i = np.flatnonzero((fd >= ts) & (fd < te))
    if len(te_i) == 0 or len(tr_i) == 0: continue
    if MAX_TRAIN_ROWS and len(tr_i) > MAX_TRAIN_ROWS:       # keep the most recent rows
        tr_i = tr_i[np.argsort(fd[tr_i], kind="stable")][-MAX_TRAIN_ROWS:]
    log(f"FOLD {name}: train {len(tr_i):,} | val {len(va_i):,} | test {len(te_i):,}")
    dtr = lgb.Dataset(X.iloc[tr_i], y[tr_i], categorical_feature=CATS, free_raw_data=True)
    dva = lgb.Dataset(X.iloc[va_i], y[va_i], reference=dtr)
    model = lgb.train(params, dtr, num_boost_round=1000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(40), lgb.log_evaluation(100)])
    pred = model.predict(X.iloc[te_i], num_iteration=model.best_iteration)
    results.append(dict(fold=name, rounds=model.best_iteration,
                        R2_raw=r2_score(y_raw[te_i], pred), R2_clip=r2_score(y[te_i], pred),
                        MAE_raw=mean_absolute_error(y_raw[te_i], pred),
                        RMSE_raw=mean_squared_error(y_raw[te_i], pred) ** 0.5,
                        base_MAE=mean_absolute_error(y_raw[te_i], np.full(len(te_i), np.median(y[tr_i])))))
    log(f"  -> R2 raw {results[-1]['R2_raw']:.3f} | R2 clip {results[-1]['R2_clip']:.3f} | MAE {results[-1]['MAE_raw']:.2f}")
    del dtr, dva; gc.collect()

res = pd.DataFrame(results).round(3)
print("\n", res.to_string(index=False))
print("\nmean R2 (raw):", round(res["R2_raw"].mean(), 3), "| mean R2 (clipped):", round(res["R2_clip"].mean(), 3))
if res["R2_raw"].max() > 0.6:
    print("WARNING: R2 above 0.6 is suspicious for this problem. Check for leakage before trusting it.")

imp = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
print("\nfeature importance (last fold, % of gain):")
print((imp / imp.sum() * 100).round(1))
# model.save_model("/kaggle/working/lgb_last_fold.txt")

2022-01-01 00:00:00 2026-08-31 00:00:00
[  0.1 min] loaded 32,957,871 rows
[  0.5 min] plane-chain features done
[  3.2 min] congestion features done
[  3.3 min] matrix ready: 32,371,463 rows x 35 features
[  3.3 min] FOLD fall_2025: train 10,000,000 | val 535,667 | test 1,718,548
Training until validation scores don't improve for 40 rounds
[100]	valid_0's rmse: 46.0249
[200]	valid_0's rmse: 45.9663
Early stopping, best iteration is:
[255]	valid_0's rmse: 45.9517
[  8.5 min]   -> R2 raw 0.339 | R2 clip 0.351 | MAE 12.79
[  8.5 min] FOLD winter_25_26: train 10,000,000 | val 520,132 | test 1,595,103
Training until validation scores don't improve for 40 rounds
[100]	valid_0's rmse: 42.8868
Early stopping, best iteration is:
[147]	valid_0's rmse: 42.8796
[ 12.0 min]   -> R2 raw 0.331 | R2 clip 0.344 | MAE 15.67
[ 12.0 min] FOLD spring_2026: train 10,000,000 | val 503,432 | test 1,792,933
Training until validation scores don't improve for 40 rounds
[100]	valid_0's rmse: 44.0033
Early stoppi